In [2]:
import pickle
import numpy as np
from matplotlib import pyplot as plt

spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]

paris_metro_colors = [
    "#FFCD00",  # Line 1 – Yellow
    "#0055C8",  # Line 2 – Blue
    "#6E6E00",  # Line 3 – Olive green
    "#CF009E",  # Line 4 – Magenta
    "#FF7E2E",  # Line 5 – Orange
    "#6ECA97",  # Line 6 – Green
    "#FA9ABA",  # Line 7 – Pink
    "#E19BDF",  # Line 8 – Lilac
    "#B6BD00",  # Line 9 – Lime green
    "#C9910D",  # Line 10 – Mustard
    "#704B1C",  # Line 11 – Brown
    "#007852",  # Line 12 – Dark green
    "#6EC4E8",  # Line 13 – Light blue
    "#62259D",  # Line 14 – Purple
]

In [77]:
pte_dict_pkl = '/scratch/gpfs/SIMONSOBS/users/md9241/LAT_ISO/plots/deep56_20260201_skn_50cg_inpaint_holes_cal_poleff/nulls/pte_dict.pkl'

file = open(pte_dict_pkl, 'rb')
pte_dict_full = pickle.load(file)
file.close()

In [ ]:
pte_dict = {key[1]: pte_dict_full[key] for key in pte_dict_full.keys() if all('220' not in k for k in key[1]) if all('280' not in k for k in key[1]) if key[0]=='lmin_1000'}

pte_dict_no_planck = {k: v for k, v in pte_dict.items() if all('planck' not in subk for subk in k)}
pte_dict_planck = {k: v for k, v in pte_dict.items() if any('planck' in subk for subk in k)}

print(pte_dict.keys())
print(pte_dict_no_planck.keys())
print(pte_dict_planck.keys())

In [ ]:
pte_list = list(pte_dict_planck.values())
label = ["Full", "LAT only", "Planck"]

n_samples = len(pte_list)
print(n_samples)
nbins = 14
bins = np.linspace(0, 1, nbins + 1)

fig, ax = plt.subplots(1, 3, figsize=(13, 4), sharex=True)
handles, labels = [], []
for i, (a, pte_list) in enumerate(zip(ax.flat, [list(pte_dict.values()), list(pte_dict_no_planck.values()), list(pte_dict_planck.values())])):
    # ax.set_title("Array-bands test", fontsize=16)
    # a.set_xlabel(r"Probability to exceed (PTE)", fontsize=16)
    a.hist(
        [pte_list],
        bins=bins,
        stacked=True,
        label=f"{label[i]} : {n_samples} tests, min: {min(pte_list):.5f}, max: {max(pte_list):.4f}",
        histtype="bar",
        facecolor=["orange", "blue"],
        edgecolor="black", 
        linewidth=3
    )
    a.axhline(len(pte_list)/nbins, color="k", ls="--", alpha=0.5)
    a.set_xlim(0, 1)
    h, l = a.get_legend_handles_labels()
    handles += h
    labels += l
    a.text(0.5, .97, label[i], transform=a.transAxes, verticalalignment='center', bbox=dict(                    # background box
        boxstyle='round',
        facecolor="white",
        alpha=1
    ))
fig.legend(handles, labels, fontsize=16, bbox_to_anchor=(.9, 0.5), loc='center left')

In [ ]:
for title, pte_dict_to_test in zip(["Full", "LAT only", "With Planck"], [pte_dict, pte_dict_no_planck, pte_dict_planck]):
    pte_list_per_spec = {
        spec: [pte 
        for k, pte in pte_dict_to_test.items()
        if spec == k[0]]
        for spec in spectra
    }

    handles, labels = [], []
    fig, ax = plt.subplots(3, 3, figsize=(10, 8), gridspec_kw={'hspace':0.1, 'wspace':.2}, sharex=True)
    for s, (a, spec) in enumerate(zip(ax.flat, spectra)):
        a.hist(
            [pte_list_per_spec[spec]],
            bins=bins,
            stacked=True,
            label=[f"{spec} n tests: {len(pte_list_per_spec[spec])}, min: {min(pte_list_per_spec[spec]):.3f}, max: {max(pte_list_per_spec[spec]):.3f}"],
            histtype="bar",
            facecolor=[f"C{s}"],
            edgecolor="black", 
            linewidth=3
        )
        a.axhline(len(pte_list_per_spec[spec])/nbins, color="k", ls="--", alpha=0.5)
        a.set_xlim(0, 1)
        h, l = a.get_legend_handles_labels()
        handles += h
        labels += l
        a.text(0.5, .97, spec, transform=a.transAxes, verticalalignment='center', bbox=dict(                    # background box
            boxstyle='round',
            facecolor="white",
            alpha=1
        ))
    fig.suptitle(title, fontsize=18)
    fig.legend(handles, labels, fontsize=16, bbox_to_anchor=(.9, 0.5), loc='center left')

In [ ]:
pte_list_per_tube_comb = {
    ('i1',): [],
    ('i3',): [],
    ('i4',): [],
    ('i6',): [],
    ('i1', 'i3'): [],
    ('i1', 'i4'): [],
    ('i1', 'i6'): [],
    ('i3', 'i4'): [],
    ('i3', 'i6'): [],
    ('i4', 'i6'): [],
    ('i1', 'i3', 'i4'): [],
    ('i1', 'i3', 'i6'): [],
    ('i1', 'i4', 'i6'): [],
    ('i1', 'i6'): [],
    ('i3', 'i4'): [],
    ('i3', 'i4', 'i6'): [],
    ('i3', 'i6'): [],
    ('i4', 'i6'): [],
    ('i1', 'i3', 'i4', 'i6'): [],
}

# TODO : remove this mess for smthg better
for k, pte in pte_dict_no_planck.items():
    tube_list = tuple([tube for tube in tubes if any([tube in svar for svar in k[1:]])])   # Quite proud of this one liner
    pte_list_per_tube_comb[tube_list] += [pte]

handles, labels = [], []
fig, ax = plt.subplots(4, 4, figsize=(10, 8), gridspec_kw={'hspace':0.1, 'wspace':.2}, sharex=True)
tubes = ['i1', 'i3', 'i4', 'i6']
for t, (a, tube_tpl) in enumerate(zip(ax.flat, pte_list_per_tube_comb.keys())):
    
    n_bins_plot = min([nbins, len(pte_list_per_tube_comb[tube_tpl]) // 2])
    bins = np.linspace(0, 1, n_bins_plot + 1)
    a.hist(
        [pte_list_per_tube_comb[tube_tpl]],
        bins=n_bins_plot,
        stacked=True,
        label=[f"{' x '.join(list(tube_tpl))} : {len(pte_list_per_tube_comb[tube_tpl])} tests, min: {min(pte_list_per_tube_comb[tube_tpl]):.5f}, max: {max(pte_list_per_tube_comb[tube_tpl]):.5f}"],
        histtype="bar",
        facecolor=['orange'],
        edgecolor="black", 
        linewidth=3
    )
    a.axhline(len(pte_list_per_tube_comb[tube_tpl])/nbins, color="k", ls="--", alpha=0.5)
    a.set_xlim(0, 1)    
    
    h, l = a.get_legend_handles_labels()
    handles += h
    labels += l
    a.text(0.5, .97, ' x '.join(list(tube_tpl)), transform=a.transAxes, verticalalignment='center', horizontalalignment='center', bbox=dict(                    # background box
        boxstyle='round',
        facecolor="white",
        alpha=1
    ))

fig.legend(handles, labels, fontsize=16, bbox_to_anchor=(.9, 0.5), loc='center left')

In [ ]:
pte_list_per_tube_comb = {
    ('i1',): [],
    ('i3',): [],
    ('i4',): [],
    ('i6',): [],
    # ('planck'): [], # THis one is empty
    ('i1', 'i3'): [],
    ('i1', 'i4'): [],
    ('i1', 'i6'): [],
    ('i1', 'planck'): [],
    ('i3', 'i4'): [],
    ('i3', 'i6'): [],
    ('i3', 'planck'): [],
    ('i4', 'i6'): [],
    ('i4', 'planck'): [],
    ('i6', 'planck'): [],
    ('i1', 'i3', 'i4'): [],
    ('i1', 'i3', 'i6'): [],
    ('i1', 'i3', 'planck'): [],
    ('i1', 'i4', 'i6'): [],
    ('i1', 'i4', 'planck'): [],
    ('i1', 'i6', 'planck'): [],
    ('i3', 'i4', 'i6'): [],
    ('i3', 'i4', 'planck'): [],
    ('i3', 'i6', 'planck'): [],
    ('i4', 'i6', 'planck'): [],
    ('i1', 'i3', 'i4', 'i6'): [],
    ('i1', 'i3', 'i4', 'planck'): [],
    ('i1', 'i3', 'i6', 'planck'): [],
    ('i1', 'i4', 'i6', 'planck'): [],
    ('i3', 'i4', 'i6', 'planck'): [],
}
tubes_w_planck = tubes + ["planck"]


# TODO : remove this mess for smthg better
for k, pte in pte_dict.items():
    tube_list = tuple([tube for tube in tubes_w_planck if any([tube in svar for svar in k[1:]])])   # Quite proud of this one liner
    # print(k, tube_list)
    pte_list_per_tube_comb[tube_list] += [pte]

handles, labels = [], []
fig, ax = plt.subplots(5, 6, figsize=(16, 12), gridspec_kw={'hspace':0.1, 'wspace':.3}, sharex=True, dpi=250)
tubes = ['i1', 'i3', 'i4', 'i6']
for t, (a, tube_tpl) in enumerate(zip(ax.flat, pte_list_per_tube_comb.keys())):
    
    n_bins_plot = min([nbins, len(pte_list_per_tube_comb[tube_tpl]) // 2])
    bins = np.linspace(0, 1, n_bins_plot + 1)
    a.hist(
        [pte_list_per_tube_comb[tube_tpl]],
        bins=n_bins_plot,
        stacked=True,
        label=[f"{' x '.join(list(tube_tpl))} : {len(pte_list_per_tube_comb[tube_tpl])} tests, min: {min(pte_list_per_tube_comb[tube_tpl]):.5f}, max: {max(pte_list_per_tube_comb[tube_tpl]):.5f}"],
        histtype="bar",
        facecolor=['orange'],
        edgecolor="black", 
        linewidth=3
    )
    a.axhline(len(pte_list_per_tube_comb[tube_tpl])/nbins, color="k", ls="--", alpha=0.5)
    a.set_xlim(0, 1)    
    
    h, l = a.get_legend_handles_labels()
    handles += h
    labels += l
    a.text(0.5, .97, ' x '.join(list(tube_tpl)), transform=a.transAxes, verticalalignment='center', horizontalalignment='center', fontsize=8, bbox=dict(                    # background box
        boxstyle='round',
        facecolor="white",
        alpha=1
    ))

fig.legend(handles, labels, fontsize=14, bbox_to_anchor=(.9, 0.5), loc='center left')